In [1]:
!pip install pennylane

In [2]:
import pennylane as qml
from pennylane import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression

In [3]:
twosides_path = "../data/raw/TWOSIDES.csv"

cols = [
    "drug_1_concept_name",
    "drug_2_concept_name",
    "condition_concept_name",
    "PRR"
]

twosides = pd.read_csv(
    twosides_path,
    usecols=cols
)

# MUCH smaller for quantum
twosides = twosides.sample(n=5000, random_state=42)

C:\Users\Kritika Taank\AppData\Local\Temp\ipykernel_25140\3866207967.py:10: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  twosides = pd.read_csv(


In [4]:
twosides["PRR"] = pd.to_numeric(
    twosides["PRR"],
    errors="coerce"
)

twosides = twosides.dropna()

def risk_label(prr):
    if prr < 1.5:
        return 0
    elif prr < 3:
        return 1
    else:
        return 2

twosides["label"] = twosides["PRR"].apply(risk_label)

In [5]:
d1 = LabelEncoder()
d2 = LabelEncoder()
eff = LabelEncoder()

twosides["d1"] = d1.fit_transform(
    twosides["drug_1_concept_name"]
)

twosides["d2"] = d2.fit_transform(
    twosides["drug_2_concept_name"]
)

twosides["eff"] = eff.fit_transform(
    twosides["condition_concept_name"]
)

X = twosides[["d1", "d2", "eff"]].values
y = twosides["label"].values

In [6]:
scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

In [7]:
n_qubits = 3

dev = qml.device("default.qubit", wires=n_qubits)

In [8]:
@qml.qnode(dev)
def quantum_circuit(inputs):

    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [9]:
quantum_features = np.array(
    [quantum_circuit(x) for x in X_scaled]
)

print(quantum_features.shape)

(5000, 3)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    quantum_features,
    y,
    test_size=0.2,
    random_state=42
)

clf = LogisticRegression(max_iter=1000)

clf.fit(X_train, y_train)

preds = clf.predict(X_test)

print("Quantum-enhanced Accuracy:")
print(accuracy_score(y_test, preds))

Quantum-enhanced Accuracy:
0.667
